In [ ]:
import subprocess
import re

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

# 1. Read current MLflow MinIO credentials from Juju
out = sh("juju run mlflow-server/0 get-minio-credentials")

access_key = re.search(r"access-key:\s*(.+)", out).group(1).strip()
secret_key = re.search(r"secret-access-key:\s*(.+)", out).group(1).strip()

print("Access key:", access_key)
print("Secret key loaded.")

# 2. Patch notebook MLflow artifact secret
sh(f"""
microk8s kubectl patch secret mlflow-server-minio-artifact \
  -n admin \
  --type merge \
  -p '{{"stringData":{{"AWS_ACCESS_KEY_ID":"{access_key}","AWS_SECRET_ACCESS_KEY":"{secret_key}"}}}}'
""")

# 3. Patch KServe S3 secret
sh(f"""
microk8s kubectl patch secret kserve-controller-s3 \
  -n admin \
  --type merge \
  -p '{{"stringData":{{"AWS_ACCESS_KEY_ID":"{access_key}","AWS_SECRET_ACCESS_KEY":"{secret_key}"}}}}'
""")

# 4. Verify
v1 = sh("microk8s kubectl get secret -n admin mlflow-server-minio-artifact -o jsonpath='{.data.AWS_SECRET_ACCESS_KEY}' | base64 -d")
v2 = sh("microk8s kubectl get secret -n admin kserve-controller-s3 -o jsonpath='{.data.AWS_SECRET_ACCESS_KEY}' | base64 -d")

print("mlflow-server-minio-artifact:", "OK" if v1 == secret_key else "MISMATCH")
print("kserve-controller-s3:", "OK" if v2 == secret_key else "MISMATCH")